# Datasets: versioned collections of Groups

A **Dataset** is a versioned (`slug/version`), freezable collection whose items are whole
**Groups of one immutable `GroupType`** - for example a dataset of `garment` groups.
Dataset-to-group membership lives in Postgres. An image-level `datasets` field is then
derived from it (Dataset -> Group -> image) and kept in OpenSearch, so images are filterable
by dataset.

What this notebook covers:

| | |
|---|---|
| [1](#1.-Connect)-[3](#3.-Groups-that-actually-hold-images) | connect, set up GroupTypes and Groups |
| [4](#4.-Create-a-Dataset---slugs-auto-version) | create a dataset; slugs auto-version |
| [5](#5.-Add-groups,-then-read-the-members-back) | add groups, read the members back |
| [6](#6.-The-type-is-immutable,-and-mismatches-are-rejected) | the `type` is immutable, and mismatched groups are rejected |
| [7](#7.-Remove-soft-deletes;-re-adding-revives)-[9](#9.-Copy-into-a-new-same-type-dataset) | remove/revive, freeze/unfreeze, copy |
| [10](#10.-Finding-datasets) | find datasets (`slug` / `type` / `search`) |
| [11](#11.-Filtering-groups) | **filter groups** - every filter, and how they combine |
| [12](#12.-Expanding-groups---one-hydrated-request) | **expand groups** - every `include_*` flag, one hydrated request |
| [13](#13.-Filtering-images-by-dataset) | **filter images by dataset** - all six dataset filters |
| [14](<#14.-Datasets-of-loose-images-(single_image)>) | `single_image` datasets: collections of loose images |
| [15](#15.-Bulk-import-at-scale) | bulk import at scale, with timings |
| [16](#16.-Cleanup) | cleanup - bulk delete, the counterpart of bulk create |
| [17](#17.-Timings) | timing summary - what the numbers mean |

## 1. Connect

The API is grouped by resource - `client.datasets.*`, `client.groups.*`, `client.images.*`,
`client.roles.*`, `client.group_types.*`. The same calls work on `DataRoomClientSync` (blocking)
and on `DataRoomClient` (awaitable) - the bulk section 15 uses the async side.


In [1]:
import os
import time
import uuid
from contextlib import contextmanager

from dataroom_client import DataRoomClientSync, DataRoomError

os.environ["DATAROOM_API_KEY"] = 'YOUR_KEY_HERE'
os.environ["DATAROOM_API_URL"] = 'http://localhost:8000/api/'

client = DataRoomClientSync()
try:
    client.images.list(limit=1)
except DataRoomError as e:
    raise RuntimeError('API token rejected. Check DATAROOM_API_KEY.') from e
print('connected to', client.api_url, ' -  token ok')

# Everything this notebook creates is suffixed with RUN, so you can re-run it against
# the same instance as often as you like without colliding with the last run.
RUN = uuid.uuid4().hex[:6]
print('run id:', RUN)

connected to http://localhost:8000/api/  -  token ok
run id: 1e81b7


### Two helpers used throughout

`timed` records how long each call takes (section 16 prints the table).

`why` unwraps a `DataRoomError` to the server's validation message.

One thing to know before reading on: the image-level `datasets` field lives in
**OpenSearch**, and OpenSearch only makes a write visible to *search* on its refresh
interval (~1s) - not the instant the write returns. So a search issued in the same breath
as the write that feeds it can legitimately come back empty. Reads addressed **by id**
(`images.get`) are immediate; only search lags. Code that must read its own writes should
retry in a small loop, exactly like sections 13 and 15 do.

In [2]:
TIMINGS = []


@contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    ms = (time.perf_counter() - start) * 1000
    TIMINGS.append((label, ms))
    print(f'    [{ms:7.0f} ms]  {label}')


def why(exc):
    """The server's validation message, not httpx's status line."""
    response = getattr(exc, 'response', None)
    return response.text if response is not None else str(exc)

## 2. Two GroupTypes

`garment` is what our dataset collects. `accessory` is a *different* type, used in section 6
to show that mismatched groups are rejected.

Roles and GroupTypes are create-once: a create against an existing name is a 400, which we
treat as "already there". Each needs its own `try` - a role that already exists must not
skip the group type that follows it.

In [3]:
PERMISSIVE_SCHEMA = {'type': 'object', 'additionalProperties': True}


def ensure_role(name, description):
    try:
        client.roles.create(name=name, description=description)
        print('  created role', name)
    except DataRoomError:
        print('  role exists', name)


def ensure_group_type(**spec):
    try:
        client.group_types.create(**spec)
        print('  created group_type', spec['name'])
    except DataRoomError:
        # GroupTypes are immutable - the name is baked into the OpenSearch encoding of
        # every group of that type, so there is no PUT or PATCH. A clash is a no-op.
        print('  group_type exists', spec['name'])


for name, description in {'flat_lay': 'Flat lay shot', 'worn_front': 'On model, front'}.items():
    ensure_role(name, description)

ensure_group_type(name='garment', description='A clothing item.', metadata_schema=PERMISSIVE_SCHEMA,
                  roles=[{'role': 'flat_lay', 'is_required': False},
                         {'role': 'worn_front', 'is_required': False}])
ensure_group_type(name='accessory', description='A non-garment accessory.', metadata_schema=PERMISSIVE_SCHEMA,
                  roles=[{'role': 'flat_lay', 'is_required': False}])

  role exists flat_lay
  role exists worn_front
  group_type exists garment


  group_type exists accessory


## 3. Groups that actually hold images

The groups must hold images, or there is nothing to filter on in section 13: the image-level
`datasets` field is derived from `Dataset -> Group -> image`, so a group with no images
contributes nothing to it.

Half the garments get both roles; half get only `flat_lay`. That asymmetry is what makes the
`has_role` filter in section 11 show something.

In [4]:
image_ids = []
for img in client.images.list(limit=12):
    image_ids.append(img['id'])
if len(image_ids) < 10:
    raise SystemExit('This instance needs at least 10 images. Run `manage.py import_images` first.')

# Four garment groups. Garments 0 and 2 also get a worn_front shot - see has_role in §11.
with timed('create 4 groups, one at a time (groups.upsert)'):
    g0 = client.groups.upsert(name=f'garment-{RUN}-0', type='garment', description='Garment #0',
                              metadata={'season': 'spring'},
                              members=[{'image_id': image_ids[0], 'role': 'flat_lay'},
                                       {'image_id': image_ids[1], 'role': 'worn_front'}])
    g1 = client.groups.upsert(name=f'garment-{RUN}-1', type='garment', description='Garment #1',
                              metadata={'season': 'spring'},
                              members=[{'image_id': image_ids[2], 'role': 'flat_lay'}])
    g2 = client.groups.upsert(name=f'garment-{RUN}-2', type='garment', description='Garment #2',
                              metadata={'season': 'spring'},
                              members=[{'image_id': image_ids[4], 'role': 'flat_lay'},
                                       {'image_id': image_ids[5], 'role': 'worn_front'}])
    g3 = client.groups.upsert(name=f'garment-{RUN}-3', type='garment', description='Garment #3',
                              metadata={'season': 'spring'},
                              members=[{'image_id': image_ids[6], 'role': 'flat_lay'}])

garment_ids = [g0['id'], g1['id'], g2['id'], g3['id']]
for g in (g0, g1, g2, g3):
    print(f"  {g['name']}  {g['image_count']} image(s)")

# One accessory group - used only for the type-mismatch demo in §6.
accessory = client.groups.upsert(name=f'accessory-{RUN}', type='accessory', description='An accessory',
                                metadata={}, members=[{'image_id': image_ids[9], 'role': 'flat_lay'}])
print('accessory group:', accessory['name'])

    [    586 ms]  create 4 groups, one at a time (groups.upsert)
  garment-1e81b7-0  2 image(s)
  garment-1e81b7-1  1 image(s)
  garment-1e81b7-2  2 image(s)
  garment-1e81b7-3  1 image(s)
accessory group: accessory-1e81b7


## 4. Create a Dataset - slugs auto-version

`POST /datasets/`. Creating against an existing slug is **not** an error: it mints the next
version. Datasets are snapshots you version, not documents you mutate in place.

In [5]:
SLUG = f'garments-{RUN}'

with timed('datasets.create'):
    ds = client.datasets.create(name='Spring Garments', slug=SLUG, type='garment',
                               description='A curated set of garment groups.')
SV = ds['slug_version']
print('created', SV, '| type =', ds['type'], '| group_count =', ds['group_count'])

# The same slug again -> version 2, side by side with version 1.
v2 = client.datasets.create(name='Spring Garments (v2)', slug=SLUG, type='garment')
print('same slug again ->', v2['slug_version'])
print('all versions of this slug:', [d['slug_version'] for d in client.datasets.list(slug=SLUG)])
print('\nWe use', SV, 'from here on.')

    [     59 ms]  datasets.create
created garments-1e81b7/1 | type = garment | group_count = 0
same slug again -> garments-1e81b7/2
all versions of this slug: ['garments-1e81b7/2', 'garments-1e81b7/1']

We use garments-1e81b7/1 from here on.


## 5. Add groups, then read the members back

`datasets.add_groups` takes group ids. Reading the members back is just the groups list filtered
by dataset - `groups.list(dataset=...)`, the mirror of `images.list(datasets=[...])` on the image
side. The dataset itself carries the total as `group_count`.

In [6]:
with timed('datasets.add_groups (4 groups)'):
    result = client.datasets.add_groups(SV, garment_ids)
print('added:', result['updated_count'], '| group_count:', client.datasets.get(SV)['group_count'])

with timed('groups.list(dataset=SV)'):
    members = client.groups.list(dataset=SV)
for g in members:
    print(f"  {g['name']:24s} type={g['type']}")

# Adding a group that is already a member is a no-op, not a duplicate and not an error.
again = client.datasets.add_groups(SV, garment_ids[:1])
print('re-add updated_count (expect 0):', again['updated_count'])

    [     70 ms]  datasets.add_groups (4 groups)


added: 4 | group_count: 4
    [     19 ms]  groups.list(dataset=SV)
  garment-1e81b7-3         type=garment
  garment-1e81b7-2         type=garment
  garment-1e81b7-1         type=garment
  garment-1e81b7-0         type=garment
re-add updated_count (expect 0): 0


## 6. The type is immutable, and mismatches are rejected

Three ways the type rule bites, all of them a 400 rather than a silent no-op:

1. `PATCH`ing a dataset's `type`,
2. adding a group whose type differs from the dataset's,
3. opening a **new version** of an existing slug with a different type - a slug's type is
   fixed by its first version, so version 2 cannot change what the collection is *of*.

In [7]:
try:
    client.datasets.update(SV, type='accessory')
    print('BUG: type change was accepted')
except DataRoomError as e:
    print('1. type change rejected  ->', why(e))

# Editing anything else is fine, and the type stays put.
patched = client.datasets.update(SV, description='edited')
print('   (description updated:', repr(patched['description']), '| type still', patched['type'], ')\n')

try:
    client.datasets.add_groups(SV, [accessory['id']])
    print('BUG: mismatched group was accepted')
except DataRoomError as e:
    print('2. mismatched group rejected  ->', why(e))

try:
    client.datasets.create(name='Wrong type', slug=SLUG, type='accessory')
    print('BUG: new version with a different type was accepted')
except DataRoomError as e:
    print('\n3. new version with a different type rejected  ->', why(e))

1. type change rejected  -> {"type":["A dataset's type is immutable and cannot be changed after creation."]}


   (description updated: 'edited' | type still garment )



2. mismatched group rejected  -> {"group_ids":"groups not of type garment: 15c74a28-fbd6-4f30-9292-b0e1c9341131"}

3. new version with a different type rejected  -> {"type":["Dataset 'garments-1e81b7' is of type 'garment'. Every version of a slug shares one type, so a new version cannot be 'accessory'. Use a different slug."]}


## 7. Remove soft-deletes; re-adding revives

Removing a group soft-deletes its membership row. Re-adding the same group **revives that row**
rather than creating a duplicate - so a group's history in a dataset stays a single row.

In [8]:
victim = garment_ids[0]
removed = client.datasets.remove_groups(SV, [victim])
print('removed:', removed['updated_count'], '| group_count now:', client.datasets.get(SV)['group_count'])

revived = client.datasets.add_groups(SV, [victim])
print('revived:', revived['updated_count'], '| group_count now:', client.datasets.get(SV)['group_count'])

removed: 1 | group_count now: 3


revived: 1 | group_count now: 4


## 8. Freeze / unfreeze

A frozen dataset rejects every membership change until it is unfrozen. This is how you publish
a version: freeze it, and it cannot drift underneath whoever is training on it.

In [9]:
client.datasets.freeze(SV)
try:
    client.datasets.add_groups(SV, garment_ids)
    print('BUG: frozen dataset accepted a write')
except DataRoomError as e:
    print('blocked while frozen  ->', why(e))

client.datasets.unfreeze(SV)
ok = client.datasets.add_groups(SV, garment_ids)
print('after unfreeze, updated_count:', ok['updated_count'], '(0 - they were all still members)')

blocked while frozen  -> ["Dataset is frozen"]


after unfreeze, updated_count: 0 (0 - they were all still members)


## 9. Copy into a new same-type dataset

`copy` clones the active memberships into a brand-new dataset that inherits the source's type.
The rows are copied inside Postgres in a single `INSERT ... SELECT`, so the cost of a copy is
almost entirely the OpenSearch write that stamps the new dataset onto the member images  - 
see the scale numbers in section 15.

In [10]:
with timed('datasets.copy (4 groups)'):
    copy = client.datasets.copy(SV, name='Spring Garments (copy)', slug=f'garments-copy-{RUN}')
COPY_SV = copy['slug_version']
print('copied to', COPY_SV, '| type =', copy['type'], '| group_count =', copy['group_count'])
print('the copy holds the same groups:',
      sorted(g['name'] for g in client.groups.list(dataset=COPY_SV)) ==
      sorted(g['name'] for g in client.groups.list(dataset=SV)))

    [     78 ms]  datasets.copy (4 groups)
copied to garments-copy-1e81b7/1 | type = garment | group_count = 4


the copy holds the same groups: True


## 10. Finding datasets

`datasets.list` lists every version, newest first within a slug, and filters by `slug`, `type`
or a free-text `search` over slug / slug_version / name.

In [11]:
print('by slug   :', [d['slug_version'] for d in client.datasets.list(slug=SLUG)])
print('by type   :', len(client.datasets.list(type='garment')), 'garment dataset(s) on this instance')
print('by search :', [d['slug_version'] for d in client.datasets.list(search=RUN)])
print()
for d in client.datasets.list(search=RUN):
    print(f"  {d['slug_version']:24s} {d['name']:26s} groups={d['group_count']}  frozen={d['is_frozen']}")

by slug   : ['garments-1e81b7/2', 'garments-1e81b7/1']


by type   : 13 garment dataset(s) on this instance
by search : ['garments-1e81b7/2', 'garments-1e81b7/1', 'garments-copy-1e81b7/1']

  garments-1e81b7/2        Spring Garments (v2)       groups=0  frozen=False
  garments-1e81b7/1        Spring Garments            groups=4  frozen=False
  garments-copy-1e81b7/1   Spring Garments (copy)     groups=4  frozen=False


## 11. Filtering groups

`groups.list` is the single entry point for fetching groups, from whatever angle. The filters
compose - every one of these is a query param, and they AND together.

| filter | matches |
|---|---|
| `type=` | groups of this GroupType |
| `dataset=` | groups that are active members of this dataset |
| `name_prefix=` | name starts with (case-insensitive) |
| `search=` | substring of name or id |
| `roles=[...]` | groups whose **type declares** all these roles |
| `has_role=[...]` | groups that **actually have** an active membership for each role |

`roles` vs `has_role` is the subtle one: `roles` asks what the type *allows*, `has_role` asks
what the group *has*. Half our garments have no `worn_front` image, so they match `roles` but
not `has_role`.

In [12]:
mine = dict(name_prefix=f'garment-{RUN}')  # scope to this run's groups

print('type=garment, this run          :', len(client.groups.list(type='garment', **mine)))
print('dataset=SV                      :', len(client.groups.list(dataset=SV)))
print('search=<run id>                 :', len(client.groups.list(search=RUN)), '(garments + the accessory)')
print()
print("roles=['worn_front']            :", len(client.groups.list(roles=['worn_front'], **mine)),
      '<- the TYPE declares worn_front, so all 4 match')
print("has_role=['worn_front']         :", len(client.groups.list(has_role=['worn_front'], **mine)),
      '<- only the ones that really have that image')
print("has_role=['flat_lay','worn_front']:", len(client.groups.list(has_role=['flat_lay', 'worn_front'], **mine)),
      '<- ALL of them, not any')
print()
# Filters compose: members of this dataset that actually have a worn_front shot.
both = client.groups.list(dataset=SV, has_role=['worn_front'])
print('dataset=SV AND has_role=worn_front:', [g['name'] for g in both])

type=garment, this run          : 4


dataset=SV                      : 4


search=<run id>                 : 5 (garments + the accessory)

roles=['worn_front']            : 4 <- the TYPE declares worn_front, so all 4 match


has_role=['worn_front']         : 2 <- only the ones that really have that image


has_role=['flat_lay','worn_front']: 2 <- ALL of them, not any



dataset=SV AND has_role=worn_front: ['garment-1e81b7-2', 'garment-1e81b7-0']


## 12. Expanding groups - one hydrated request

Everything above fetched thin groups. The same list call can hydrate them instead. Each flag
adds **one page-scoped join or one batched OpenSearch read** - never a per-row round trip, so
the cost is flat in the size of the page, not in the number of groups.

| flag | adds |
|---|---|
| `include_roles` | `roles: [{image_id, role, metadata}, ...]` |
| `return_roles=[...]` | narrows `roles` to those role names |
| `include_metadata` | the group's own metadata blob |
| `include_thumbnail_urls` | a `thumbnail_url` per member image |
| `include_presigned_urls` | a `presigned_url` per member image - the **original**, or `null`. It never quietly falls back to the thumbnail |
| `include_os_metadata` | each member image's `attributes` from OpenSearch |
| `include_datasets` | the datasets each group belongs to |

The url and os_metadata flags imply `include_roles` (they hang off the members). So a dataset's
entire contents - groups, their images, roles, metadata and urls - come back in one paginated
call.

In [13]:
with timed('groups.list (thin)'):
    client.groups.list(dataset=SV)

with timed('groups.list (fully hydrated)'):
    full = client.groups.list(
        dataset=SV,
        include_roles=True,
        include_metadata=True,
        include_thumbnail_urls=True,
        include_presigned_urls=True,
        include_os_metadata=True,
        include_datasets=True,
    )

for g in full[:2]:
    print(f"\n{g['name']}  type={g['type']}  metadata={g.get('metadata')}")
    print(f"  datasets: {g.get('datasets')}")
    for m in g.get('roles', []):
        os_meta = m.get('os_metadata')
        attrs = sorted(os_meta)[:6] if os_meta else '(this image has no attributes)'
        print(f"    {m['role']:12s} {m['image_id'][:13]}...  membership metadata={m['metadata']}")
        print(f"    {'':12s}   thumbnail_url = {(m.get('thumbnail_url') or ' - ')[:56]}")
        print(f"    {'':12s}   presigned_url = {(m.get('presigned_url') or ' -  (image has no original)')[:56]}")
        print(f"    {'':12s}   os_metadata   = {attrs}")

    [     21 ms]  groups.list (thin)
    [     27 ms]  groups.list (fully hydrated)

garment-1e81b7-3  type=garment  metadata={'season': 'spring'}
  datasets: ['garments-1e81b7/1', 'garments-copy-1e81b7/1']
    flat_lay     bench_0_103...  membership metadata={}
                   thumbnail_url =  - 
                   presigned_url = http://localhost:9000/dataroom-local/images/bench_0_103/
                   os_metadata   = (this image has no attributes)

garment-1e81b7-2  type=garment  metadata={'season': 'spring'}
  datasets: ['garments-1e81b7/1', 'garments-copy-1e81b7/1']
    flat_lay     bench_0_101...  membership metadata={}
                   thumbnail_url =  - 
                   presigned_url = http://localhost:9000/dataroom-local/images/bench_0_101/
                   os_metadata   = (this image has no attributes)
    worn_front   bench_0_102...  membership metadata={}
                   thumbnail_url =  - 
                   presigned_url = http://localhost:9000/dataroom-loc

In [14]:
# return_roles narrows the expansion to the roles you care about - useful when a type has
# many roles and you only want one of them hydrated.
only_flat = client.groups.list(dataset=SV, include_roles=True, return_roles=['flat_lay'])
for g in only_flat:
    print(f"{g['name']:24s} roles kept: {[m['role'] for m in g['roles']]}")

garment-1e81b7-3         roles kept: ['flat_lay']
garment-1e81b7-2         roles kept: ['flat_lay']
garment-1e81b7-1         roles kept: ['flat_lay']
garment-1e81b7-0         roles kept: ['flat_lay']


## 13. Filtering images by dataset

An image is "in" a dataset when it belongs to a Group the dataset contains. That two-hop
relation is denormalised onto each image's `datasets` field in OpenSearch and kept in sync on
every membership change - so images are filterable by dataset without touching Postgres.

| filter | matches images that |
|---|---|
| `datasets=[a, b]` | are in **any** of these (OR) |
| `datasets__all=[a, b]` | are in **all** of these (AND) |
| `datasets__ne=[a]` | are **not** in any of these |
| `datasets__ne_all=[a, b]` | are not in *all* of these (i.e. missing at least one) |
| `datasets__prefix=[slug]` | are in **any version** of this slug |
| `datasets__empty=True` | are in **no** dataset at all |

To tell the OR filters from the AND filters, we need two datasets that genuinely *differ*  - 
so first make a second one holding only **half** the groups. (The section-9 copy is no use here:
it holds exactly the same groups as its source, so its intersection with the source is also its
union, and every filter would return the same number.)

Note the retry loop: these are *searches*, so they only see the membership write after
OpenSearch's next refresh (~1s). `images.get` is addressed by id and is immediate.

In [15]:
# A second dataset with only the first two garment groups - a strict subset of SV.
half = client.datasets.create(name='Half the garments', slug=f'garments-half-{RUN}', type='garment')
HALF_SV = half['slug_version']
client.datasets.add_groups(HALF_SV, garment_ids[:2])
print(f'{SV:24s} holds {len(garment_ids)} groups')
print(f'{HALF_SV:24s} holds 2 of them\n')

# Addressed by id - immediate, no refresh needed.
sample = image_ids[0]
detail = client.images.get(sample, include_fields=['datasets'])
print('images.get(...) datasets:', detail.get('datasets'), '\n')

# Searches - poll until the denorm write is visible (see §1 on eventual consistency).
with timed('images.count(datasets=[SV]) - incl. refresh wait'):
    n_sv = client.images.count(datasets=[SV])
    for attempt in range(60):
        if n_sv > 0:
            break
        time.sleep(0.5)
        n_sv = client.images.count(datasets=[SV])

n_half = client.images.count(datasets=[HALF_SV])
for attempt in range(60):
    if n_half > 0:
        break
    time.sleep(0.5)
    n_half = client.images.count(datasets=[HALF_SV])
print(f'datasets=[SV]                  -> {n_sv:6d}  images in the full dataset')
print(f'datasets=[HALF]                -> {n_half:6d}  images in the half dataset\n')

print(f'datasets=[SV, HALF]            -> {client.images.count(datasets=[SV, HALF_SV]):6d}  '
      'in EITHER (union - same as SV, every HALF image is in SV)')
print(f'datasets__all=[SV, HALF]       -> {client.images.count(datasets__all=[SV, HALF_SV]):6d}  '
      'in BOTH (intersection - only the half)')
print(f'datasets__ne=[HALF]            -> {client.images.count(datasets__ne=[HALF_SV]):6d}  '
      'in NEITHER of the listed')
print(f'datasets__ne_all=[SV, HALF]    -> {client.images.count(datasets__ne_all=[SV, HALF_SV]):6d}  '
      'missing at least one of them')
print(f'datasets__prefix=[{SLUG}] -> {client.images.count(datasets__prefix=[SLUG]):6d}  '
      'any version of the slug')

empty = client.images.count(datasets__empty=True)
print(f'datasets__empty=True           -> {empty:6d}  in no dataset at all'
      f"{'  (0 - every image on this instance is already in some dataset)' if not empty else ''}")

garments-1e81b7/1        holds 4 groups
garments-half-1e81b7/1   holds 2 of them



images.get(...) datasets: ['garments-1e81b7/1', 'garments-copy-1e81b7/1', 'garments-half-1e81b7/1', 'log-d1212e/1', 'loose-09e7c9/1', 'loose-5f5f1a/1'] 



    [     55 ms]  images.count(datasets=[SV]) - incl. refresh wait
datasets=[SV]                  ->      6  images in the full dataset
datasets=[HALF]                ->      3  images in the half dataset



datasets=[SV, HALF]            ->      6  in EITHER (union - same as SV, every HALF image is in SV)


datasets__all=[SV, HALF]       ->      3  in BOTH (intersection - only the half)


datasets__ne=[HALF]            -> 103007  in NEITHER of the listed
datasets__ne_all=[SV, HALF]    -> 103007  missing at least one of them


datasets__prefix=[garments-1e81b7] ->      6  any version of the slug


datasets__empty=True           ->     49  in no dataset at all


In [16]:
# Dataset filters are ordinary image filters: they compose with every other one.
# "Images in this dataset, from this source, at least 100px on the short edge":
combined = client.images.list(datasets=[SV], sources=['squares'], short_edge__gte=100,
                            include_fields=['datasets'], limit=5)
print(len(combined), 'image(s) matched dataset + source + size')
for img in combined[:3]:
    print(f"  {img['id'][:13]}...  datasets={img.get('datasets')}")

0 image(s) matched dataset + source + size


## 14. Datasets of loose images (`single_image`)

Datasets collect Groups - so a dataset of *individual images* uses the `single_image` GroupType:
a group wrapping exactly one image (role `single_image`), reused one per image. You never build
those groups yourself; `datasets.add_images` wraps each image for you.

The GroupType has to exist before the dataset is created, because `type` is resolved by name at
creation and is immutable afterwards. Register it once, like any other type.

The call is idempotent: re-adding an image already in the dataset revives its membership instead
of duplicating the group. It rejects a frozen dataset, or a dataset of any other type.

In [17]:
ensure_role('single_image', 'The single image in a group.')
ensure_group_type(name='single_image', description='A group wrapping exactly one image.',
                  metadata_schema=PERMISSIVE_SCHEMA,
                  roles=[{'role': 'single_image', 'is_required': True}])

loose = client.datasets.create(name='Loose Shots', slug=f'loose-{RUN}', type='single_image')
LOOSE_SV = loose['slug_version']
loose_ids = [img['id'] for img in client.images.list(limit=5)]

with timed('datasets.add_images (5 loose images)'):
    added = client.datasets.add_images(LOOSE_SV, loose_ids)
print('added:', added)
print('re-adding the same images (idempotent):', client.datasets.add_images(LOOSE_SV, loose_ids))
print('group_count:', client.datasets.get(LOOSE_SV)['group_count'], ' -  one wrapper group per image')

# Adding images to a dataset that is NOT single_image is a 400.
try:
    client.datasets.add_images(SV, loose_ids)
    print('BUG: a garment dataset accepted loose images')
except DataRoomError as e:
    print('rejected on a non-single_image dataset  ->', why(e))

  role exists single_image


  group_type exists single_image


    [     76 ms]  datasets.add_images (5 loose images)
added: {'updated_count': 5}


re-adding the same images (idempotent): {'updated_count': 0}
group_count: 5  -  one wrapper group per image
rejected on a non-single_image dataset  -> ["Only 'single_image' datasets accept images; this dataset is of type 'garment'."]


## 15. Bulk import at scale

The realistic test: take a whole `source` and turn it into a dataset.

Doing it a group at a time - `groups.upsert` per group - is one HTTP round trip, one transaction
and one OpenSearch write **each**. `groups.create_many` builds up to 1000 groups per request: one
transaction, one OpenSearch write. The cell below measures both on the same instance, so the
ratio is real rather than quoted.

`MAX_IMAGES = None` takes the whole source. `GROUP_SIZE` is the lever on group count, hence on
request count and wall clock.
The bulk cells run on the async `DataRoomClient` - the same calls await there
too, so it is `await aclient.groups.create_many(...)` with the exact same signature.


In [18]:
import asyncio

from dataroom_client import DataRoomClient

SOURCE = 'squares'    # a source that exists on your instance
GROUP_SIZE = 10       # images per group
MAX_IMAGES = 20_000   # None = the whole source
BULK = 1000           # groups per groups.create_many request (the server's cap)
CONCURRENCY = 4

ensure_role('member', 'A member image of a batch group.')
ensure_group_type(name='flux_batch', description='An arbitrary batch of images.',
                  metadata_schema=PERMISSIVE_SCHEMA,
                  roles=[{'role': 'member', 'is_required': False}])

aclient = DataRoomClient()

with timed(f'fetch image ids from source={SOURCE!r}'):
    ids = [img['id'] async for img in aclient.images.iter(
        sources=[SOURCE], fields=['id'], limit=MAX_IMAGES, page_size=2000)]
print(f'  {len(ids)} images')
if not ids:
    raise SystemExit(f'No images with source={SOURCE!r}. Import some, or pick another source.')

  role exists member


  group_type exists flux_batch


    [    686 ms]  fetch image ids from source='squares'
  20000 images


In [19]:
# --- baseline: one group per request -------------------------------------------------
SAMPLE = 20
t = time.perf_counter()
for i in range(SAMPLE):
    members = []
    for image_id in ids[i * GROUP_SIZE:(i + 1) * GROUP_SIZE]:
        members.append({'image_id': image_id, 'role': 'member'})
    client.groups.upsert(name=f'slow-{RUN}-{i}', type='flux_batch', members=members)
per_group_slow = (time.perf_counter() - t) / SAMPLE * 1000
print(f'groups.upsert, one at a time : {per_group_slow:7.1f} ms/group')

# --- bulk: up to 1000 groups per request ----------------------------------------------
specs = []
for i in range(0, len(ids), GROUP_SIZE):
    members = []
    for image_id in ids[i:i + GROUP_SIZE]:
        members.append({'image_id': image_id, 'role': 'member'})
    specs.append({'name': f'flux-{RUN}-{i}', 'type': 'flux_batch', 'members': members})
sem = asyncio.Semaphore(CONCURRENCY)


async def create_chunk(chunk):
    async with sem:
        return await aclient.groups.create_many(chunk)


chunks = []
for i in range(0, len(specs), BULK):
    chunks.append(specs[i:i + BULK])
print(f'{len(specs)} groups of <={GROUP_SIZE} images, in {len(chunks)} bulk request(s)')

with timed(f'groups.create_many - {len(specs)} groups'):
    t = time.perf_counter()
    tasks = []
    for chunk in chunks:
        tasks.append(create_chunk(chunk))
    results = await asyncio.gather(*tasks)
    created = []
    for result in results:
        for g in result:
            created.append(g)
    build = time.perf_counter() - t
per_group_bulk = build / len(created) * 1000
print(f'groups.create_many, in bulk      : {per_group_bulk:7.1f} ms/group  '
      f'-> {per_group_slow / per_group_bulk:.0f}x faster ({build / len(ids) * 1000:.2f} ms/image)')

groups.upsert, one at a time :   151.2 ms/group
2000 groups of <=10 images, in 2 bulk request(s)


    [  15465 ms]  groups.create_many - 2000 groups
groups.create_many, in bulk      :     7.7 ms/group  -> 20x faster (0.77 ms/image)


In [20]:
# Collect them into a dataset, 100 group ids per request.
perf = await aclient.datasets.create(name=f'Perf {SOURCE}', slug=f'perf-{RUN}', type='flux_batch')
PSV = perf['slug_version']
gids = []
for g in created:
    gids.append(g['id'])

# Bounded like the create phase: each chunk triggers a ~1000-image OpenSearch write,
# so firing all of them at once can overflow the write queue on a small instance.
async def add_chunk(chunk):
    async with sem:
        return await aclient.datasets.add_groups(PSV, chunk)


with timed(f'datasets.add_groups - {len(gids)} groups'):
    tasks = []
    for i in range(0, len(gids), 100):
        tasks.append(add_chunk(gids[i:i + 100]))
    res = await asyncio.gather(*tasks)
added = 0
for r in res:
    added = added + r['updated_count']
print(f'  added {added} groups to {PSV}')

# The copy path is the one tuned in this branch: the membership rows are duplicated with a
# single INSERT..SELECT inside Postgres, so what is left is the OpenSearch write.
with timed(f'datasets.copy - {len(gids)} groups'):
    perf_copy = await aclient.datasets.copy(PSV, name=f'Perf {SOURCE} (copy)', slug=f'perf-copy-{RUN}')

print(f"\ngroup_count: {(await aclient.datasets.get(PSV))['group_count']}")
with timed('images.count(datasets=[PSV]) - incl. refresh wait'):
    n = client.images.count(datasets=[PSV])
    for attempt in range(120):
        if n >= len(ids):
            break
        time.sleep(0.5)
        n = client.images.count(datasets=[PSV])
print(f'images carrying {PSV}: {n}')
await aclient.client.aclose()

    [  11225 ms]  datasets.add_groups - 2000 groups
  added 2000 groups to perf-1e81b7/1


    [  13552 ms]  datasets.copy - 2000 groups

group_count: 2000
    [     20 ms]  images.count(datasets=[PSV]) - incl. refresh wait
images carrying perf-1e81b7/1: 20000


## 16. Cleanup

Everything this run created is suffixed with the run id, so it is easy to find and easy to drop.

Two things worth watching here:

- **Deleting a dataset also strips its `slug_version` from its member images.** Otherwise they
  would stay filterable by a dataset that no longer exists - and since the membership rows go
  with the dataset, nothing would be left for the reconciler to notice.
- **`groups.delete_many` is the counterpart of `groups.create_many`** - up to 1000 ids per request, one
  transaction and one OpenSearch write, versus one round trip *per group* with `groups.delete`.
  The timing table below puts the two side by side; the ratio is the same as it is on the way in.

In [21]:
CLEANUP = True

if not CLEANUP:
    print('skipped - everything is tagged with run id', RUN)
else:
    with timed('delete datasets'):
        for sv in [d['slug_version'] for d in client.datasets.list(search=RUN)]:
            client.datasets.delete(sv)
            print('  deleted dataset', sv)

    stale = [g['id'] for g in client.groups.list(search=RUN, limit=100_000)]

    # The old way, for comparison: one request per group.
    one_at_a_time, stale = stale[:20], stale[20:]
    t = time.perf_counter()
    for gid in one_at_a_time:
        client.groups.delete(gid)
    per_group_slow = (time.perf_counter() - t) / len(one_at_a_time) * 1000

    # The bulk way: up to 1000 ids per request.
    t = time.perf_counter()
    with timed(f'groups.delete_many - {len(stale)} groups in bulk'):
        deleted = sum(
            client.groups.delete_many(stale[i:i + 1000])['deleted_count']
            for i in range(0, len(stale), 1000)
        )
    per_group_bulk = (time.perf_counter() - t) / max(deleted, 1) * 1000

    print(f'\ndelete_group, one at a time : {per_group_slow:7.1f} ms/group')
    print(f'groups.delete_many, in bulk      : {per_group_bulk:7.1f} ms/group  '
          f'-> {per_group_slow / per_group_bulk:.0f}x faster ({deleted} deleted)')

  deleted dataset garments-1e81b7/2


  deleted dataset garments-1e81b7/1
  deleted dataset garments-copy-1e81b7/1


  deleted dataset garments-half-1e81b7/1


  deleted dataset loose-1e81b7/1


  deleted dataset perf-1e81b7/1


  deleted dataset perf-copy-1e81b7/1
    [  28308 ms]  delete datasets


    [  19212 ms]  groups.delete_many - 2005 groups in bulk

delete_group, one at a time :   111.8 ms/group
groups.delete_many, in bulk      :     9.6 ms/group  -> 12x faster (2005 deleted)


## 17. Timings

What the numbers should look like, and why:

- **Per-group writes are round-trip bound.** `groups.upsert` / `groups.delete` one at a time is one
  HTTP request, one transaction and one OpenSearch write *each*, so each lands in the
  tens-to-hundreds of ms and barely improves with a faster server. `groups.create_many` and
  `groups.delete_many` amortise all three over a batch of up to 1000 and land ~1-2 ms/group - two
  orders of magnitude, and the gap widens with scale. Reach for the bulk pair whenever you are
  moving more than a handful.
- **`datasets.copy` is OpenSearch bound.** Postgres duplicates the membership rows in a single
  `INSERT ... SELECT`, so what is left is the bulk write stamping the new `slug_version` onto
  every member image. Expect it to track *image* count, not group count - copying 2000 groups
  costs about what adding them did.
- **Reads are flat.** A hydrated `groups.list` costs about the same as a thin one: each `include_*`
  flag adds one page-scoped join or one batched OpenSearch read, not a per-row round trip.
- **`images.count` timings include the refresh wait**, so on a cold write they are dominated by
  OpenSearch's ~1s refresh interval rather than the query. A search on a warm index is
  single-digit ms.

In [22]:
width = max(len(label) for label, _ in TIMINGS)
print(f"{'step':<{width}}   {'ms':>9}")
print('─' * (width + 12))
for label, ms in TIMINGS:
    print(f'{label:<{width}}   {ms:9.0f}')

step                                                       ms
─────────────────────────────────────────────────────────────
create 4 groups, one at a time (groups.upsert)            586
datasets.create                                            59
datasets.add_groups (4 groups)                             70
groups.list(dataset=SV)                                    19
datasets.copy (4 groups)                                   78
groups.list (thin)                                         21
groups.list (fully hydrated)                               27
images.count(datasets=[SV]) - incl. refresh wait           55
datasets.add_images (5 loose images)                       76
fetch image ids from source='squares'                     686
groups.create_many - 2000 groups                        15465
datasets.add_groups - 2000 groups                       11225
datasets.copy - 2000 groups                             13552
images.count(datasets=[PSV]) - incl. refresh wait          20
delete d